# 04 - Baseline CNN Training (optional, bridge to Phase 2)

Uses the outputs of `03_preprocessing.ipynb` (`eye_dataset_64.npz`, `splits.csv`, `normalization_stats.json`) to train a small CNN that predicts whether an eye is **closed** (target `closed`: 1 = closed, 0 = open).

| Decision | Reason |
|---|---|
| Subject-wise train / val / test from notebook 03 | No person appears in two splits, so the test score is honest |
| Class weights from `normalization_stats.json` | Computed from the training set only |
| Light augmentation (brightness, contrast, small shift / zoom, flip) | EDA showed large lighting / sensor differences; augmentation only runs during training |
| Threshold chosen on the **validation** set | The test set is used once, at the end |
| Metrics: recall and F1 of the *closed* class, AUC | Missing a closed eye is the costly error for drowsiness; accuracy alone hides that |

Outputs: `models/eye_state_cnn.keras`, `models/threshold.json`, `models/test_metrics.json`, figures in `images/train_*.png`.

In [ ]:
import os, json, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
sns.set_theme(style='whitegrid')
PROC_DIR, IMG_DIR, MODEL_DIR = '../data/processed', '../images', '../models'
for d in (IMG_DIR, MODEL_DIR): os.makedirs(d, exist_ok=True)
EPOCHS, BATCH = 25, 64                   # lower EPOCHS (e.g. 3) for a quick test run
def savefig(name): plt.savefig(os.path.join(IMG_DIR, name), dpi=150, bbox_inches='tight')
print('TensorFlow', tf.__version__, '| GPU:', bool(tf.config.list_physical_devices('GPU')))

## 1. Load the processed data
Images are stored as `uint8`; we scale to [0, 1] exactly as in notebook 03. `splits.csv` + `metadata_clean.csv` give each test image its lighting / glasses / sensor, so we can evaluate **per condition** later.

In [ ]:
d = np.load(f'{PROC_DIR}/eye_dataset_64.npz')
Xtr, ytr = d['X_train'].astype('float32') / 255, d['y_train'].astype('int32')
Xva, yva = d['X_val'].astype('float32') / 255,   d['y_val'].astype('int32')
Xte, yte = d['X_test'].astype('float32') / 255,  d['y_test'].astype('int32')
stats = json.load(open(f'{PROC_DIR}/normalization_stats.json'))
class_weights = {int(k): v for k, v in stats['class_weights'].items()}

splits = pd.read_csv(f'{PROC_DIR}/splits.csv')
meta = pd.read_csv(f'{PROC_DIR}/metadata_clean.csv')[['rel_path', 'gender', 'glasses', 'reflections', 'lighting', 'sensor']]
test_meta = splits[splits.split == 'test'].merge(meta, on='rel_path', how='left').reset_index(drop=True)
assert len(test_meta) == len(yte) and (test_meta.closed.values == yte).all(), 'test metadata is not aligned with the test arrays'

print('train', Xtr.shape, '| val', Xva.shape, '| test', Xte.shape)
print('class weights:', {k: round(v, 3) for k, v in class_weights.items()})

## 2. Model
Four convolution blocks with batch normalisation, global average pooling and dropout. The output is the probability that the eye is closed. The augmentation layers are part of the model but are only active while training.

In [ ]:
L = tf.keras.layers
def build_model():
    return tf.keras.Sequential([
        L.Input(shape=(64, 64, 1)),
        L.RandomFlip('horizontal'),
        L.RandomTranslation(0.08, 0.08, fill_mode='nearest'),
        L.RandomZoom(0.10, fill_mode='nearest'),
        L.RandomBrightness(0.25, value_range=(0, 1)),
        L.RandomContrast(0.25),
        L.Conv2D(32, 3, padding='same', activation='relu'), L.BatchNormalization(), L.MaxPooling2D(),
        L.Conv2D(64, 3, padding='same', activation='relu'), L.BatchNormalization(), L.MaxPooling2D(),
        L.Conv2D(128, 3, padding='same', activation='relu'), L.BatchNormalization(), L.MaxPooling2D(),
        L.Conv2D(128, 3, padding='same', activation='relu'), L.BatchNormalization(), L.GlobalAveragePooling2D(),
        L.Dropout(0.4), L.Dense(128, activation='relu'), L.Dropout(0.3),
        L.Dense(1, activation='sigmoid'),
    ])

model = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(3e-4), loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
                       tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')])
model.summary()

## 3. Train
Early stopping on validation AUC restores the best weights; the learning rate is halved when validation loss stalls.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
]
history = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS, batch_size=BATCH,
                    class_weight=class_weights, callbacks=callbacks, verbose=2)
model.save(f'{MODEL_DIR}/eye_state_cnn.keras')
h = pd.DataFrame(history.history)
h.to_csv(f'{PROC_DIR}/training_history.csv', index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
h[['loss', 'val_loss']].plot(ax=ax[0], title='Loss'); h[['auc', 'val_auc']].plot(ax=ax[1], title='AUC')
for a in ax: a.set_xlabel('epoch')
plt.tight_layout(); savefig('train_01_curves.png'); plt.show()

**What it tells us:** if training loss keeps falling while validation loss rises, the model is overfitting. Validation curves that follow the training curves closely mean the model generalises to unseen people.

## 4. Choose the decision threshold on the validation set

In [ ]:
pva = model.predict(Xva, batch_size=256, verbose=0).ravel()
thresholds = np.arange(0.10, 0.91, 0.01)
f1s = [f1_score(yva, (pva >= t).astype(int), zero_division=0) for t in thresholds]
THR = float(thresholds[int(np.argmax(f1s))])
json.dump({'threshold': THR}, open(f'{MODEL_DIR}/threshold.json', 'w'))
print(f'Threshold from validation data: {THR:.2f} (validation F1 = {max(f1s):.4f})')

## 5. Final evaluation on the untouched test set

In [ ]:
pte = model.predict(Xte, batch_size=256, verbose=0).ravel()
yhat = (pte >= THR).astype(int)
metrics = {'threshold': THR, 'accuracy': accuracy_score(yte, yhat),
           'precision_closed': precision_score(yte, yhat, zero_division=0),
           'recall_closed': recall_score(yte, yhat, zero_division=0),
           'f1_closed': f1_score(yte, yhat, zero_division=0),
           'auc': roc_auc_score(yte, pte), 'test_images': int(len(yte))}
test_n = metrics.pop('test_images')
metrics = {k: round(float(v), 4) for k, v in metrics.items()}; metrics['test_images'] = int(test_n)
json.dump(metrics, open(f'{MODEL_DIR}/test_metrics.json', 'w'), indent=2)
print(pd.Series(metrics).to_string())

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.heatmap(confusion_matrix(yte, yhat, labels=[0, 1]), annot=True, fmt='d', cmap='Blues', ax=ax[0],
            xticklabels=['open', 'closed'], yticklabels=['open', 'closed']); ax[0].set(xlabel='Predicted', ylabel='Actual', title='Confusion matrix')
fpr, tpr, _ = roc_curve(yte, pte); ax[1].plot(fpr, tpr); ax[1].plot([0, 1], [0, 1], '--', color='gray')
ax[1].set(title=f"ROC curve (AUC {metrics['auc']:.3f})", xlabel='False positive rate', ylabel='True positive rate')
pr, rc, _ = precision_recall_curve(yte, pte); ax[2].plot(rc, pr); ax[2].set(title='Precision-recall curve', xlabel='Recall', ylabel='Precision')
plt.tight_layout(); savefig('train_02_test_evaluation.png'); plt.show()

## 6. Where does the model fail? Results per recording condition
A good overall score can hide a weak group (for example bad lighting or glasses). Groups with few images give noisy numbers, so check the `n` column.

In [ ]:
test_meta['correct'] = (yhat == yte)
test_meta['pred'] = yhat
rows = []
for col, names in [('lighting', {0: 'bad light', 1: 'good light'}), ('glasses', {0: 'no glasses', 1: 'glasses'}),
                   ('reflections', {0: 'no reflection', 1: 'small', 2: 'big'}), ('sensor', {1: 'sensor 1', 2: 'sensor 2', 3: 'sensor 3'}),
                   ('gender', {0: 'male', 1: 'female'})]:
    for v, g in test_meta.groupby(col):
        closed = g[g.closed == 1]
        rows.append({'condition': col, 'group': names.get(v, v), 'n': len(g), 'accuracy': g.correct.mean(),
                     'recall_closed': (closed.pred == 1).mean() if len(closed) else np.nan})
by_cond = pd.DataFrame(rows).round(3)
by_cond.to_csv(f'{PROC_DIR}/test_metrics_by_condition.csv', index=False)
print(by_cond.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=by_cond.assign(label=by_cond.condition + ': ' + by_cond.group.astype(str)), y='label', x='recall_closed', color='indianred', ax=ax)
ax.axvline(metrics['recall_closed'], color='k', ls='--', lw=.8, label='overall'); ax.legend(); ax.set_xlim(0, 1)
ax.set_title('Recall of closed eyes per condition'); ax.set_ylabel('')
plt.tight_layout(); savefig('train_03_recall_by_condition.png'); plt.show()

per_subject = test_meta.groupby('subject').correct.mean()
print(f'\nPer-subject accuracy: min {per_subject.min():.3f} | median {per_subject.median():.3f} | max {per_subject.max():.3f}')

## Summary
* Baseline CNN trained on the subject-wise split from notebook 03; the threshold was tuned on validation data only.
* Test metrics are saved in `models/test_metrics.json`; per-condition results in `data/processed/test_metrics_by_condition.csv`.
* Phase 2 can start from here: try a deeper model or transfer learning, tune hyper-parameters, and compare against this baseline.